In [19]:
import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import json
import matplotlib.patches as mpatches


### New output format for L shaped results

In [20]:
import json
import pandas as pd

# Collect all parsed trial frames here
df_full_list = []

for trial in range(1,12):
    # Load the JSON file
    filename = f"results/MVP_DE_results_T_10_delta_5_scen_1_trial_{trial}_inv_1_cap._1_cap.inc._1.json"
    with open(filename) as json_file:
        data = json.load(json_file)

    # Step 1: Extract rows from the 'F' dictionary
    rows = []
    for antigen, sub_dict in data['F'].items():
        for starting, ending_dict in sub_dict.items():
            for ending, fill in ending_dict.items():
                fill_value = 'Full' if fill == 1.0 else 'None'
                rows.append([antigen, int(starting), int(ending), fill_value])

    df_filtered_F = pd.DataFrame(rows, columns=['Antigen', 'Start', 'Finish', 'Fill'])

    # Step 2: Filter for Full only
    df_filtered_F = df_filtered_F[df_filtered_F['Fill'] != 'None'].copy()

    # Step 3: Overlap detection
    for antigen, group in df_filtered_F.groupby('Antigen'):
        sorted_group = group.sort_values(by='Start')
        for i in range(len(sorted_group) - 1):
            current_row = sorted_group.iloc[i]
            next_row = sorted_group.iloc[i + 1]
            if current_row['Finish'] >= next_row['Start']:
                df_filtered_F.loc[current_row.name, 'Fill'] = 'Overlap'
                df_filtered_F.loc[next_row.name, 'Fill'] = 'Overlap'

    # Step 4: Apply initial fill and shift start values
    df_filtered_F.loc[df_filtered_F['Start'] == 1, 'Fill'] = 'Initial'
    df_filtered_F['Start'] = df_filtered_F['Start'] - 1

    # Step 5: Tag the trial number as time step
    df_filtered_F["Time"] = trial

    # Add to list
    df_full_list.append(df_filtered_F)

# Step 6: Combine all
df_full_copy = pd.concat(df_full_list, ignore_index=True)


In [21]:
import plotly.graph_objects as go
import pandas as pd

fig = go.Figure()

# Mappings
opacity_map = {"Overlap": 0.65, "Initial": 0.1, "Full": 1.0}
colors = {
    "MMR Antigens": "blue",
    "Hexa Antigens": "green",
    "HPV": "red",
    "PCV": "purple",
    "Rota": "orange",
}
group_map = {antigen: group for group, antigens in {
    "MMR Antigens": ["Measles", "Mumps", "Rubella"],
    "Hexa Antigens": ["Diphtheria", "Tetanus", "Pertussis", "Hib", "Polio", "Hepatitis_B"],
    "HPV": ["HPV"],
    "PCV": ["PCV"],
    "Rota": ["Rotavirus"],
}.items() for antigen in antigens}

# Prepare dataframe
df = df_full_copy.copy()
df["Group"] = df["Antigen"].map(group_map)
df["Color"] = df["Group"].map(colors)
df["Width"] = df["Finish"] - df["Start"]

# Unique values
time_steps = sorted(df["Time"].unique())
fill_types = df["Fill"].unique()
frames = []

# Build traces and frames
for t in time_steps:
    df_t = df[df["Time"] == t]
    frame_data = []

    for fill in fill_types:
        df_tf = df_t[df_t["Fill"] == fill]
        trace = go.Bar(
            x=df_tf["Width"],
            y=df_tf["Antigen"],
            base=df_tf["Start"],
            orientation='h',
            marker=dict(
                color=df_tf["Color"],
                line=dict(width=1.5, color="black")
            ),
            opacity=opacity_map.get(fill, 1.0),
            name=fill,
            customdata=df_tf[["Start", "Finish", "Group"]],
            hovertemplate="Antigen: %{y}<br>Start: %{customdata[0]}<br>Finish: %{customdata[1]}<br>Group: %{customdata[2]}<extra></extra>"
        )
        if t == time_steps[0]:
            fig.add_trace(trace)  # Add only the first frame to main fig
        frame_data.append(trace)

    frames.append(go.Frame(data=frame_data, name=str(t)))

# Animation layout
fig.update_layout(
    updatemenus=[dict(
        type="buttons",
        showactive=False,
        buttons=[dict(label="Play",
                      method="animate",
                      args=[None, {
                          "frame": {"duration": 1000, "redraw": True},
                          "fromcurrent": True,
                          "transition": {"duration": 300}}])]
    )],
    sliders=[dict(
        steps=[dict(method="animate",
                    args=[[str(t)], {"frame": {"duration": 800, "redraw": True}, "mode": "immediate"}],
                    label=str(t-1)) for t in time_steps],
        transition={"duration": 200},
        x=0, y=0, currentvalue=dict(font=dict(size=14), prefix="Demand Increase (%): ", visible=True),
        len=1.0
    )],
    barmode='overlay',
    title="Interactive Tender Schedule Over Time - MP Model",
    xaxis=dict(title="Time", range=[0, 10]),
    yaxis=dict(title="Antigen", categoryorder='array', categoryarray=df["Antigen"].unique()[::-1]),
    showlegend=True
)

fig.frames = frames
fig.write_html("MP_interactive_chart.html", include_plotlyjs='cdn', full_html=True)

fig.show()
